# Particle I/O and Catalogue Conversion

This notebook explains the data boundary immediately upstream of `SFCProjection`. PyHermes readers turn local files or HTTP(S) resources into one shared dictionary with `pos` of shape `(N, 3)`, `size`, and any requested one-dimensional particle fields.

Start with `quick_start.ipynb` for the shortest complete calculation. Return here when adapting a catalogue of your own.

## What this notebook emphasizes

- URL download, checksum verification, and cache reuse
- NPZ field discovery and field renaming
- writing a portable NPZ catalogue
- reading a raw BIN table with an explicit column layout
- when to use the FoF, Gadget, and Gadget HDF5 readers


In [ ]:
from pathlib import Path
import os

import numpy as np
import yaml

from pyhermes.io import read_particle_data
from pyhermes.param.parambase import read_param

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "examples":
    examples_dir = cwd.parent
elif cwd.name == "examples":
    examples_dir = cwd
elif (cwd / "examples").is_dir():
    examples_dir = cwd / "examples"
else:
    raise RuntimeError("Run this notebook from the project root, examples/, or examples/notebooks/.")

os.chdir(examples_dir)
data_dir = Path("./data")
data_dir.mkdir(exist_ok=True)
print(f"Working directory: {Path.cwd()}")


## 1. Read the public URL catalogue

The Quick Start projection YAML is the source of truth for the public catalogue URL, local cache path, and SHA256 checksum. The first call downloads the file; later calls use the verified cache without contacting the server.


In [ ]:
projection_config = read_param("./configs/param_sfc_projection.yaml")["SFCProjection"]
catalog_fin = projection_config["fin"]

halo_data = read_particle_data(
    catalog_fin["path"],
    data_format=catalog_fin["format"],
    download=catalog_fin["download"],
)

print(f"particles: {halo_data['size']:,}")
print(f"positions: {halo_data['pos'].shape} {halo_data['pos'].dtype}")
print(f"fields: {sorted(key for key in halo_data if key not in {'pos', 'size'})}")


The NPZ arrays carry names and dtypes. Scientific provenance and units live in the small tracked schema beside the ignored data file.


In [ ]:
schema = yaml.safe_load(Path("./data/quijote_halos_8000_snap004_schema.yaml").read_text())
print(schema["source"])
for name, metadata in schema["arrays"].items():
    print(f"{name:>6s}: shape={metadata['shape']}, dtype={metadata['dtype']}, units={metadata['units']}")


## 2. Select and rename particle fields

`fields` maps the names used by your analysis to names stored in the file. Positions are always returned as `pos`; the other fields are opt-in when a mapping is supplied.


In [ ]:
halo_selected = read_particle_data(
    catalog_fin["path"],
    data_format="npz",
    download=catalog_fin["download"],
    fields={
        "vx": "vel_x",
        "vy": "vel_y",
        "vz": "vel_z",
        "mass": "mass",
        "npart": "npart",
    },
)
print(sorted(halo_selected))


## 3. Write a portable NPZ catalogue

NPZ is a convenient interchange format because array names, shapes, and dtypes travel with the file. This example writes a small subset so the tutorial remains lightweight; remove the slice to convert the complete catalogue.


In [ ]:
subset_count = 50_000
subset_path = data_dir / "quijote_halos_subset.npz"
np.savez_compressed(
    subset_path,
    pos=np.asarray(halo_selected["pos"][:subset_count], dtype=np.float32),
    vel_x=np.asarray(halo_selected["vx"][:subset_count], dtype=np.float32),
    vel_y=np.asarray(halo_selected["vy"][:subset_count], dtype=np.float32),
    vel_z=np.asarray(halo_selected["vz"][:subset_count], dtype=np.float32),
    mass=np.asarray(halo_selected["mass"][:subset_count], dtype=np.float32),
    npart=np.asarray(halo_selected["npart"][:subset_count], dtype=np.float32),
)

subset_data = read_particle_data(subset_path, data_format="npz")
np.testing.assert_allclose(subset_data["pos"], halo_selected["pos"][:subset_count])
print(f"Wrote and reloaded {subset_path} with {subset_data['size']:,} particles")


## 4. Read a raw binary table

A raw BIN file stores bytes but no column names, shape, or units. Its reader parameters are therefore part of the analysis contract. Here the layout is defined next to the conversion code instead of being hidden in a downloaded data directory.


In [ ]:
bin_count = 10_000
bin_path = data_dir / "quijote_halos_subset.bin"
bin_table = np.column_stack(
    [
        halo_selected["pos"][:bin_count],
        halo_selected["vx"][:bin_count],
        halo_selected["vy"][:bin_count],
        halo_selected["vz"][:bin_count],
        halo_selected["mass"][:bin_count],
        halo_selected["npart"][:bin_count],
    ]
).astype(np.float32, copy=False)
bin_table.tofile(bin_path)

bin_reader_params = {
    "dtype": "float32",
    "ncols": 8,
    "pos_cols": [0, 1, 2],
    "fields": {
        "vel_x": 3,
        "vel_y": 4,
        "vel_z": 5,
        "mass": 6,
        "npart": 7,
    },
}
bin_data = read_particle_data(bin_path, data_format="bin", **bin_reader_params)
np.testing.assert_allclose(bin_data["pos"], halo_selected["pos"][:bin_count])
print(f"Wrote and reloaded {bin_path} with shape {bin_table.shape}")


## 5. Native simulation readers

PyHermes also reads formats that are inconvenient to package inside a public notebook. Use these readers directly when the original simulation data are available locally or on shared storage.

### Quijote/Pylians FoF directory

The optional preparation command downloads the original example archive and can recreate the public NPZ catalogue:

```bash
python scripts/prepare_sfc_fields.py --catalog-only
```

```python
fof_data = read_particle_data(
    "./data/quijote_halos/8000",
    data_format="fof",
    snapnum=4,
    redshift=0.0,
    fields={"mass": "mass", "vel": "vel"},
)
```

### Gadget snapshots

Use `format="gadget"` for legacy binary snapshots, `format="gadget_hdf5"` for HDF5 snapshots, and `format="gadget-fof"` for legacy Gadget FoF catalogues. Split files are discovered from their shared base path. Position, velocity, and mass conversion factors belong in `reader_params`; see the I/O reference for the complete options.


## 6. Hand the catalogue to `SFCProjection`

Every route above ends with the same objects: positions plus optional catalogue weights and field values. The next notebook uses exactly those arrays to build number-, mass-, and redshift-space `SFCField` objects.

```python
task.particle_pos = halo_selected["pos"]
task.field_value = halo_selected["mass"]
```
